In [20]:
import json
from pathlib import Path

In [21]:
rules_path = Path("../data/rules/contract_rules.json")

In [22]:
with open(rules_path,'r',encoding='utf-8') as f:
    rules = json.load(f)
print(type(relus))
print(len(rules))
print(rules[0]['title'])

NameError: name 'relus' is not defined

In [5]:
contract_text = """
6.3 水质超标处理费

甲方交付的污水水质超过本协议约定的接收标准时，
乙方有权按照原污水处理单价的1.5倍收取超标处理费用。
"""

In [6]:
rule = rules[0]

prompt = f"""
你是一名工业污水处理合同审查助手。

请严格依据以下审查规则，检查提供的合同条款。

审查规则：
{rule['review_question']}

审查要求：
{rule['expected_requirement']}

合同原文：
{contract_text}

请回答：
1. 该条款是否满足审查要求？
2. 找出存在问题的原文。
3. 解释具体风险。
4. 给出修改建议。

注意：
只根据提供的合同片段审查。
没有出现的条款不能认定整份合同一定缺失。
不得编造合同内容。
"""

print(prompt)


你是一名工业污水处理合同审查助手。

请严格依据以下审查规则，检查提供的合同条款。

审查规则：
合同是否明确水质超标时的处理方式、收费机制，以及严重超标情况下的价格协商机制？

审查要求：
合同应明确超标污水的拒收或接收条件、超标处理费用，以及现有计价规则不适用时的协商处理机制。

合同原文：

6.3 水质超标处理费

甲方交付的污水水质超过本协议约定的接收标准时，
乙方有权按照原污水处理单价的1.5倍收取超标处理费用。


请回答：
1. 该条款是否满足审查要求？
2. 找出存在问题的原文。
3. 解释具体风险。
4. 给出修改建议。

注意：
只根据提供的合同片段审查。
没有出现的条款不能认定整份合同一定缺失。
不得编造合同内容。



In [19]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
load_dotenv()
api_key = os.getenv("SILICONFLOW_API_KEY")
llm = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V4.1-Flash",
    base_url="https://api.siliconflow.cn/v1",
    api_key=api_key)

In [20]:
response = llm.invoke(prompt)
print(response.content)

1. **该条款是否满足审查要求？**  
   仅就该条款而言，**不满足/未完整满足审查要求**。该条款明确了超标处理费用为原单价1.5倍，部分满足“收费机制”；但**未明确超标污水的拒收或接收条件**，也**未明确严重超标或现有计价规则不适用时的价格协商机制**。不能据此认定整份合同其他条款一定缺失，但该条款本身不完整。

2. **存在问题的原文。**  
   > “甲方交付的污水水质超过本协议约定的接收标准时，乙方有权按照原污水处理单价的1.5倍收取超标处理费用。”

   问题点：仅约定超过接收标准即按1.5倍收费，未区分超标程度，未约定乙方是否可拒收及拒收/接收条件，也未约定严重超标时价格如何协商。

3. **具体风险。**  
   - **拒收/接收条件不明确**：污水超标时，乙方是否必须接收、是否有权拒收、在何种情形下拒收，条款未明确，易产生操作争议。
   - **收费机制过于单一**：只要超标即按1.5倍收费，未区分轻微超标与严重超标；严重超标导致处理成本大幅增加时，1.5倍可能不足以覆盖成本或引发价格争议。
   - **严重超标协商机制缺失**：当超标严重或现有1.5倍计价规则不适用时，缺少价格协商、补充定价及协商不成的处理路径，可能导致费用无法确定。
   - **“有权收取”表述不完整**：仅赋予乙方收费权，但未同步明确接收/拒收权及相应条件。

4. **修改建议。**  
   建议补充明确以下内容，可修改为类似条款：

   > **6.3 水质超标处理费**  
   > 6.3.1 甲方交付的污水水质超过本协议约定接收标准时，乙方有权根据超标项目、超标程度及对污水处理系统的影响，决定接收或拒收；具体拒收/接收条件为：【双方约定】。  
   > 6.3.2 乙方接收超标污水的，超标处理费用按原污水处理单价的1.5倍收取，适用条件为：【明确适用范围】。  
   > 6.3.3 发生严重超标【可约定为超过接收标准一定倍数、多项指标同时超标或导致处理系统无法正常运行等具体情形】或现有计价规则不适用时，双方应在【约定期限】内另行协商处理价格，并签订补充协议；协商不成的，按本合同争议解决条款处理。

   上述修改仅针对该条款本身；片段未体现的其他内容，不据此认定整份合同必然缺失。


In [21]:
from pydantic import BaseModel,Field
from typing import Literal

class ConstractReviewResult(BaseModel):
    rule_id:str = Field(
        description='审查规则编号'
    )
    status:Literal['符合要求','存在问题','无法判断'] = Field(
        description='依据当前提供的合同文本作出的审查判断'
    )
    evidence: str = Field(
        description="支持审查结论的合同原文，不得编造"
    )

    issue: str = Field(
        description="发现的问题及其具体原因"
    )

    suggestion: str = Field(
        description="针对问题提出的合同修改建议"
    )

In [23]:
print(ConstractReviewResult.model_json_schema())

{'properties': {'rule_id': {'description': '审查规则编号', 'title': 'Rule Id', 'type': 'string'}, 'status': {'description': '依据当前提供的合同文本作出的审查判断', 'enum': ['符合要求', '存在问题', '无法判断'], 'title': 'Status', 'type': 'string'}, 'evidence': {'description': '支持审查结论的合同原文，不得编造', 'title': 'Evidence', 'type': 'string'}, 'issue': {'description': '发现的问题及其具体原因', 'title': 'Issue', 'type': 'string'}, 'suggestion': {'description': '针对问题提出的合同修改建议', 'title': 'Suggestion', 'type': 'string'}}, 'required': ['rule_id', 'status', 'evidence', 'issue', 'suggestion'], 'title': 'ConstractReviewResult', 'type': 'object'}


In [24]:
structured_llm = llm.with_structured_output(ConstractReviewResult)

In [26]:
response = structured_llm.invoke(prompt)

ValidationError: 1 validation error for ConstractReviewResult
  Invalid JSON: invalid number at line 1 column 3 [type=json_invalid, input_value='1. **是否满足审查...协商处理机制**。', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/json_invalid

In [27]:
review_prompt = prompt + f"\n审查规则编号：{rule['rule_id']}。请按照指定的结构化字段输出。"

In [28]:
result = structured_llm.invoke(review_prompt)

print(type(result))
print(result.model_dump())

<class '__main__.ConstractReviewResult'>
{'rule_id': 'WQ-001', 'status': '存在问题', 'evidence': '6.3 水质超标处理费：甲方交付的污水水质超过本协议约定的接收标准时，乙方有权按照原污水处理单价的1.5倍收取超标处理费用。', 'issue': '该条款仅约定水质超标时按原污水处理单价1.5倍收费，未明确超标污水的拒收或接收条件，未区分严重超标情形，也未约定现有计价规则不适用时的价格协商机制，不满足审查要求。', 'suggestion': '建议补充约定：1）明确水质超标时乙方的拒收或接收条件，例如按超标项目、超标倍数、持续时长等划分可接收与拒收情形；2）明确超标处理费的适用条件和计算方式，并说明是否仅适用于轻微或一般超标；3）增加严重超标情形下的处理机制，包括停止接收、限期整改、另行协商处理价格或委托第三方处理等；4）明确协商不成的后续处理路径。'}


In [29]:
good_contract = """
6.3 水质超标处理费

甲方交付的污水超过接收标准，但超标倍数不超过2倍，
且经乙方评估确认可以处理的，
经乙方书面同意接收后，按照原污水处理单价的1.5倍收费。

当任一指标超标超过2倍，或可能影响污水处理系统安全稳定运行时，
乙方有权拒绝接收。

如双方拟继续接收该类严重超标污水，
须在接收前另行书面协商处理价格。
未能达成一致的，乙方有权拒收。
"""

In [30]:
good_prompt = review_prompt.replace(
    contract_text,
    good_contract
)

In [35]:
good_prompt += """
请严格按照以下要求生成结构化结果：

1. status 只能是：符合要求、存在问题、无法判断。
   如果部分满足但仍有明确缺失，填写“存在问题”,请严格返回字符串。

2. evidence 必须是一个字符串，不得使用列表。
   应直接引用提供的合同原文。

3. issue 必须是字符串。
   如果符合要求，填写“无”。

4. suggestion 必须是字符串。
   如果符合要求，填写“无需修改”。

5. 不得遗漏任何必填字段。
"""

In [36]:
good_result = structured_llm.invoke(good_prompt)

print(good_result.model_dump())

ValidationError: 1 validation error for ConstractReviewResult
rule_id
  Field required [type=missing, input_value={'status': '符合要求'...estion': '无需修改'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing

In [42]:
debug_llm = llm.with_structured_output(ConstractReviewResult,
                   
                                       include_raw=True)
debug_result = debug_llm.invoke(good_prompt)


In [43]:
print("解析结果：", debug_result["parsed"])
print("解析错误：", debug_result["parsing_error"])
print("原始工具调用：", debug_result["raw"].tool_calls)

解析结果： rule_id='WQ-001' status='符合要求' evidence='6.3 水质超标处理费：甲方交付的污水超过接收标准，但超标倍数不超过2倍，且经乙方评估确认可以处理的，经乙方书面同意接收后，按照原污水处理单价的1.5倍收费。当任一指标超标超过2倍，或可能影响污水处理系统安全稳定运行时，乙方有权拒绝接收。如双方拟继续接收该类严重超标污水，须在接收前另行书面协商处理价格。未能达成一致的，乙方有权拒收。' issue='无' suggestion='无需修改'
解析错误： None
原始工具调用： []


In [25]:
from pathlib import Path

test_path = Path("../data/raw/contracts/test_contract.docx")

print(test_path.exists())
print(test_path)

True
..\data\raw\contracts\test_contract.docx


In [26]:
from src.contract_review import load_contract_docx

doc_text = load_contract_docx(test_path)
print(doc_text)

污水处理服务协议

6.3 水质超标处理费

甲方交付的污水水质超过本协议约定的接收标准时，乙方有权按照原污水处理单价的1.5倍收取超标处理费用。

附件六：污水处理费用

表格内容
项目|单价
污水处理费|25元/吨


In [27]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

print("项目根目录：", PROJECT_ROOT)
print("src目录存在：", (PROJECT_ROOT / "src").exists())

项目根目录： C:\Users\Administrator\Documents\ai-study\industrial-wastewater-ai
src目录存在： True


In [28]:
from src.contract_review import load_contract_docx

doc_text = load_contract_docx(test_path)

print(doc_text)

污水处理服务协议

6.3 水质超标处理费

甲方交付的污水水质超过本协议约定的接收标准时，乙方有权按照原污水处理单价的1.5倍收取超标处理费用。

附件六：污水处理费用

表格内容
项目|单价
污水处理费|25元/吨


In [4]:
%load_ext autoreload
%autoreload 2

In [29]:
from src.contract_review import load_contract_rules

In [30]:
rules_path = PROJECT_ROOT / "data" / "rules" / "contract_rules.json"

In [31]:
from src.contract_review import load_contract_rules

In [32]:
rules = load_contract_rules(rules_path)

In [33]:
print(len(rules))
print([rule["rule_id"] for rule in rules])

3
['WQ-001', 'WQ-002', 'WQ-003']


In [34]:
from src.contract_review import ContractReviewResult

test_result = ContractReviewResult(
    rule_id="WQ-001",
    status="存在问题",
    evidence="超过接收标准时，按照1.5倍收费。",
    issue="未明确严重超标情况下的价格协商机制。",
    suggestion="建议增加严重超标时的书面协商机制。"
)
print(test_result.model_dump())

{'rule_id': 'WQ-001', 'status': '存在问题', 'evidence': '超过接收标准时，按照1.5倍收费。', 'issue': '未明确严重超标情况下的价格协商机制。', 'suggestion': '建议增加严重超标时的书面协商机制。'}


In [35]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI
load_dotenv()
api_key = os.getenv("SILICONFLOW_API_KEY")
llm = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V4.1-Flash",
    base_url="https://api.siliconflow.cn/v1",
    api_key=api_key)

In [36]:
from src.contract_review import ContractReviewResult

In [37]:
structured_llm = llm.with_structured_output(ContractReviewResult,
                                           method="json_mode",
                                            include_raw=True)

print("结构化输出模型初始化成功")

结构化输出模型初始化成功


In [18]:
print(type(llm))
print(llm)

<class 'langchain_openai.chat_models.base.ChatOpenAI'>
client=<openai.resources.chat.completions.completions.Completions object at 0x000001F010045A00> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001F010493710> root_client=<openai.OpenAI object at 0x000001F00FBC2BD0> root_async_client=<openai.AsyncOpenAI object at 0x000001F00FC7F740> model_name='deepseek-ai/DeepSeek-V4.1-Flash' model_kwargs={} openai_api_key=SecretStr('**********') openai_api_base='https://api.siliconflow.cn/v1' openai_proxy=None


In [19]:
test_prompt = """
请审查以下合同条款：

甲方交付的污水水质超过本协议约定的接收标准时，
乙方有权按照原污水处理单价的1.5倍收取超标处理费用。

审查要求：
检查是否明确严重超标情况下的价格协商机制。
"""

result = structured_llm.invoke(test_prompt)

print(result)
print(type(result))

{'raw': AIMessage(content='**审查结论：该条款未明确严重超标情况下的价格协商机制，存在约定空白与履约风险。**\n\n具体分析如下：\n\n**1. 条款仅设定了单一加价标准，未区分超标程度**\n当前条款约定：只要甲方交付的污水水质“超过”约定接收标准，乙方即有权按原污水处理单价的 **1.5倍** 收费。该表述未区分一般超标与严重超标，也未设置阶梯式加价或另行协商机制。\n\n**2. 缺少“严重超标”的定义或识别标准**\n条款未明确何为“严重超标”，例如：\n- 超标项目的种类（COD、氨氮、pH、重金属等）；\n- 超标倍数或浓度阈值；\n- 是否涉及有毒有害物质；\n- 是否导致乙方处理系统受损或无法正常运行。\n\n缺乏上述标准，双方对是否构成“严重超标”容易产生争议。\n\n**3. 未约定严重超标时的价格协商或另行处理机制**\n对于严重超标情形，通常应约定：\n- 乙方有权暂停接收或要求甲方整改；\n- 双方另行协商处理价格；\n- 乙方可拒绝接收或要求甲方承担额外处置成本；\n- 因严重超标造成乙方损失的赔偿责任。\n\n但当前条款仅规定统一按1.5倍收费，未预留严重超标情形下的价格协商空间，可能导致：\n- 乙方在严重超标时仍只能按1.5倍收费，无法覆盖实际处理成本；\n- 甲方可能认为1.5倍即为封顶价格，拒绝另行协商；\n- 双方对严重超标后的费用调整缺乏合同依据。\n\n**4. 修改建议**\n建议将条款调整为分层机制，例如：\n\n> 甲方交付的污水水质超过本协议约定接收标准的，乙方有权按照原污水处理单价的1.5倍收取超标处理费用。若超标情形达到本协议约定的严重超标标准（如某一指标超过接收标准【X】倍，或含有毒有害物质，或导致乙方处理系统异常等），双方应就处理价格、整改期限及赔偿责任另行协商；协商不成的，乙方有权暂停接收超标污水，由此产生的后果由甲方承担。\n\n**审查意见：** 现有条款未明确严重超标情况下的价格协商机制，建议补充严重超标的认定标准、协商程序及乙方救济措施，以避免履约争议。', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_token

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

system_prompt = """
你是工业污水处理服务合同审查助手。

你的任务是根据合同原文和审查规则，判断合同条款是否满足要求。

必须严格遵守以下要求：

1. 只能依据合同原文进行判断，不得编造合同内容。
2. status 只能是以下三个值：
   - 符合要求
   - 存在问题
   - 无法判断
3. evidence 必须引用合同中的真实原文。
4. issue 必须说明具体问题及风险。
5. suggestion 必须提供具体、可执行的修改建议。
6. 如果符合要求，issue 填写空字符串。
7. rule_id 必须与输入的规则编号完全一致。

输出要求：
必须返回一个合法 JSON 对象。
必须包含以下五个字段：
rule_id、status、evidence、issue、suggestion。

所有字段的值均为字符串。
禁止输出 Markdown。
禁止使用代码块包裹 JSON。
禁止在 JSON 前后添加解释性文字。
"""

rule = rules[0]

human_prompt = f"""
请审查以下合同。

【审查规则】
规则编号：{rule['rule_id']}
规则名称：{rule['title']}
审查问题：{rule['review_question']}
预期要求：{rule['expected_requirement']}

【合同原文】
{contract_text}
"""

response = structured_llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content=human_prompt)
])

In [ ]:
print("解析错误：", response["parsing_error"])

print("原始输出：")
print(response["raw"].content)

print("结构化结果：")
print(response["parsed"])

In [23]:
from src.contract_review import review_contract

In [43]:
results = review_contract(
    contract_text=doc_text,
    rules=rules,
    structured_llm=structured_llm
)

for result in results:
    print("=" * 60)
    print(result)

rule_id='WQ-001' status='存在问题' evidence='6.3 水质超标处理费\n\n甲方交付的污水水质超过本协议约定的接收标准时，乙方有权按照原污水处理单价的1.5倍收取超标处理费用。' issue='合同仅约定了水质超标时按原污水处理单价的1.5倍收取超标处理费用，未明确超标污水的拒收或接收条件，也未明确严重超标或特殊水质情况下现有计价规则不适用时的价格协商机制。' suggestion='建议在合同水质超标处理条款中补充明确：1）超标污水的拒收或接收条件；2）严重超标或特殊水质情况下，现有计价规则不适用时，双方可就处理价格另行书面协商确认。涉及新增商务条件的，需双方进行确认。'
rule_id='WQ-002' status='存在问题' evidence='6.3 水质超标处理费\n\n甲方交付的污水水质超过本协议约定的接收标准时，乙方有权按照原污水处理单价的1.5倍收取超标处理费用。\n\n附件六：污水处理费用\n\n表格内容\n项目|单价\n污水处理费|25元/吨' issue='合同文本仅约定污水处理费单价及水质超标处理费，未明确污水接收水量的计量方式、计量点位、抄表周期、双方确认方式及计量争议处理机制，无法满足水量计量与结算机制审查要求。' suggestion='建议在合同中补充明确水量计量机制，包括：1.计量设备及计量点位（如进水阀门、流量计安装位置及设备管理责任）；2.水量记录和抄表周期；3.双方共同抄表、签字确认及结算周期；4.计量设备异常、数据缺失或双方对水量有异议时的复核、校验或第三方检定机制。涉及新增计量点位、计量设备责任及复核费用承担等商务条件的，需双方进行确认。'
rule_id='WQ-003' status='存在问题' evidence='6.3 水质超标处理费\n\n甲方交付的污水水质超过本协议约定的接收标准时，乙方有权按照原污水处理单价的1.5倍收取超标处理费用。\n\n附件六：污水处理费用\n\n表格内容\n项目|单价\n污水处理费|25元/吨' issue='合同仅约定了水质超标后的处理费，但未明确水质采样位置、采样频次、检测指标、检测方法、超标判定依据，以及检测结果争议处理机制。' suggestion='建议补充明确采样点位、采样频次、检测指标、检测方法、超标判定依据，以及采样记录、检测结

In [44]:
for result in results:
    print(result)

rule_id='WQ-001' status='存在问题' evidence='6.3 水质超标处理费\n\n甲方交付的污水水质超过本协议约定的接收标准时，乙方有权按照原污水处理单价的1.5倍收取超标处理费用。' issue='合同仅约定了水质超标时按原污水处理单价的1.5倍收取超标处理费用，未明确超标污水的拒收或接收条件，也未明确严重超标或特殊水质情况下现有计价规则不适用时的价格协商机制。' suggestion='建议在合同水质超标处理条款中补充明确：1）超标污水的拒收或接收条件；2）严重超标或特殊水质情况下，现有计价规则不适用时，双方可就处理价格另行书面协商确认。涉及新增商务条件的，需双方进行确认。'
rule_id='WQ-002' status='存在问题' evidence='6.3 水质超标处理费\n\n甲方交付的污水水质超过本协议约定的接收标准时，乙方有权按照原污水处理单价的1.5倍收取超标处理费用。\n\n附件六：污水处理费用\n\n表格内容\n项目|单价\n污水处理费|25元/吨' issue='合同文本仅约定污水处理费单价及水质超标处理费，未明确污水接收水量的计量方式、计量点位、抄表周期、双方确认方式及计量争议处理机制，无法满足水量计量与结算机制审查要求。' suggestion='建议在合同中补充明确水量计量机制，包括：1.计量设备及计量点位（如进水阀门、流量计安装位置及设备管理责任）；2.水量记录和抄表周期；3.双方共同抄表、签字确认及结算周期；4.计量设备异常、数据缺失或双方对水量有异议时的复核、校验或第三方检定机制。涉及新增计量点位、计量设备责任及复核费用承担等商务条件的，需双方进行确认。'
rule_id='WQ-003' status='存在问题' evidence='6.3 水质超标处理费\n\n甲方交付的污水水质超过本协议约定的接收标准时，乙方有权按照原污水处理单价的1.5倍收取超标处理费用。\n\n附件六：污水处理费用\n\n表格内容\n项目|单价\n污水处理费|25元/吨' issue='合同仅约定了水质超标后的处理费，但未明确水质采样位置、采样频次、检测指标、检测方法、超标判定依据，以及检测结果争议处理机制。' suggestion='建议补充明确采样点位、采样频次、检测指标、检测方法、超标判定依据，以及采样记录、检测结

In [64]:
from pathlib import Path
from docx import Document
def export_contract_report(
    results: list[ContractReviewResult],
    output_path: str | Path,
    contract_name: str
) -> Path:
    
    doc = Document()
    doc.add_heading(f'{contract_name}合同建议报告')
    problem_count = sum(1 for result in results
                       if result.status == "存在问题")
    unknown_count = sum(1 for result in results
                      if result.status == "无法判断")
    pass_count = sum(1 for result in results
                    if result.status =="复合要求")
    doc.add_heading("一、总体审查结论", level=1)

    doc.add_paragraph(f"本次共审查 {len(results)} 项合同规则。")
    doc.add_paragraph(f"符合要求：{pass_count} 项")
    doc.add_paragraph(f"存在问题：{problem_count} 项")
    doc.add_paragraph(f"无法判断：{unknown_count} 项")
    
    if problem_count > 0:
        conclusion = "本次审查发现部分合同条款存在问题，建议结合具体审查意见进行修改完善。"
    elif unknown_count > 0:
        conclusion = "本次审查存在无法判断的事项，建议补充相关合同资料后进一步核实。"
    else:
        conclusion = "本次审查涉及的合同规则均符合审查要求。"
    
    doc.add_paragraph(conclusion)
    for result in results:
        doc.add_heading(
        f"{result.rule_id} 审查结果",
        level=2
    )

        doc.add_paragraph(
            f"审查结论：{result.status}"
        )
    
        doc.add_heading("合同原文", level=3)
        doc.add_paragraph(result.evidence)
    
        doc.add_heading("存在问题", level=3)
        doc.add_paragraph(result.issue)
    
        doc.add_heading("修改建议", level=3)
        doc.add_paragraph(result.suggestion)
    doc.save(output_path)
        
    return Path(output_path)


In [65]:
output_path = PROJECT_ROOT / "data" / "reports" / "合同审查报告.docx"

output_path.parent.mkdir(parents=True, exist_ok=True)

report_path = export_contract_report(
    results=results,
    output_path=output_path,
    contract_name="污水处理服务协议"
)

In [66]:
print(report_path)
print(report_path.exists())

C:\Users\Administrator\Documents\ai-study\industrial-wastewater-ai\data\reports\合同审查报告.docx
True
